<a href="https://colab.research.google.com/github/PedroNunezMartinez/ProyectoModuloIA/blob/main/rv_motor_crm/agente_clasificador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ==============================================================================
# MASTER EN IA - PROYECTO FINAL: R&V MOTOR (SEMANAS 1 Y 2)
# GENERACIÓN DE DATASET SINTÉTICO Y CLASIFICACIÓN CON MODELO GEMINI 3.8 FLASH
# ==============================================================================

import json
import time
import pandas as pd
from google.genai import client
from google.genai import types
from google.colab import userdata
from google.colab import files

print("🚀 Iniciando el pipeline analítico modernizado de R&V Motor...")

# 1. CONEXIÓN SEGURA A LA API DE GEMINI (Ciberseguridad)
try:
    # Recupera la clave guardada en el menú de la llave (Secrets) de Colab
    api_key_env = userdata.get('GEMINI_API_KEY')

    # Inicializamos el cliente oficial con el SDK moderno de Google Gemini
    ai = client.Client(api_key=api_key_env)
    print("✅ Conexión segura con el nuevo SDK de Google Gemini establecida.")
except Exception as e:
    print("❌ ERROR DE CREDENCIALES: Verifica que guardaste tu API Key en los Secrets de Colab como 'GEMINI_API_KEY' y activaste el switch de acceso.")
    raise e

# ==============================================================================
# FASE 1 (SEMANA 1): GENERACIÓN AUTOMÁTICA DEL DATASET DE 50 CLIENTES
# ==============================================================================
print("\n⏳ Fase 1: Generando lote de 50 mensajes sintéticos de clientes...")

prompt_semana1 = """
Actúa como un Generador de Datos Sintéticos experto. Necesito una lista de exactamente 50 interacciones de clientes simulados para la empresa "R&V Motor", una concesionaria que vende motocicletas en América Latina.
Debes devolver ÚNICAMENTE un objeto JSON que contenga un arreglo llamado "clientes". No incluyas texto introductorio ni bloques markdown ```json.

Cada objeto debe tener esta estructura:
{
  "id_cliente": "C001",
  "nombre_cliente": "Nombre y apellido hispano realista",
  "telefono": "Número ficticio de 10 dígitos",
  "mensaje_texto": "Mensaje realista de WhatsApp preguntando por motocicletas, créditos o cotizaciones",
  "clasificacion_real_docente": "Etiqueta origen para control académico entre: 'Lead Caliente', 'Cliente Potencial', 'Lead Frío', 'Irrelevante'"
}

Distribución requerida: 15 Leads Calientes, 20 Clientes Potenciales, 10 Leads Fríos, 5 Irrelevantes.
"""

# Usamos gemini-3.8-flash, el modelo oficial exigido por la API de Google
response_s1 = ai.models.generate_content(
    model='gemini-3.8-flash',
    contents=prompt_semana1,
    config=types.GenerateContentConfig(response_mime_type="application/json")
)

datos_s1 = json.loads(response_s1.text)
df_clientes = pd.DataFrame(datos_s1["clientes"])
print("💾 Datos base de la Semana 1 generados internamente con éxito.")

# ==============================================================================
# FASE 2 (SEMANA 2): PIPELINE DEL AGENTE DE IA (CLASIFICACIÓN ESTRICTA)
# ==============================================================================
print("\n🤖 Fase 2: Configurando el Agente de IA y procesando la clasificación...")

system_instruction = """
Actúa como un Analista de Ventas y CRM Automatizado para la concesionaria de motocicletas R&V Motor.
Tu tarea es analizar el mensaje de texto del cliente y clasificarlo bajo estas reglas estrictas:
- 'Lead Caliente': Alta intención de compra inmediata (pide cuentas de depósito, transferencias, disponibilidad de entrega inmediata).
- 'Cliente Potencial': Interés real pero requiere información (pregunta por requisitos de crédito, financiamiento, fichas técnicas o citas).
- 'Lead Frío': Objeciones definitivas, respuestas vagas o desinterés por precio ("está muy caro gracias", "ok").

Debes responder ÚNICAMENTE en formato JSON con la siguiente estructura:
{
  "clasificacion_ia": "Lead Caliente" | "Cliente Potencial" | "Lead Frío",
  "motivo_analisis": "Explicación corta de una frase sobre por qué se asignó esa etiqueta",
  "modelo_interes": "Modelo de motocicleta detectado o 'No especifica'",
  "respuesta_sugerida": "Un mensaje corto, empático y comercial saludando al cliente y guiándolo al siguiente paso del embudo de ventas."
}
"""

clasificaciones_ia = []
motivos = []
modelos = []
respuestas_bot = []

print("⏳ Procesando los mensajes fila por fila con el Agente LLM...")
for index, row in df_clientes.iterrows():
    mensaje = row['mensaje_texto']
    try:
        # Petición estructurada usando el motor de última generación gemini-3.8-flash
        response_s2 = ai.models.generate_content(
            model='gemini-3.8-flash',
            contents=mensaje,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                response_mime_type="application/json"
            )
        )
        res_json = json.loads(response_s2.text)

        clasificaciones_ia.append(res_json.get("clasificacion_ia", "Lead Frío"))
        motivos.append(res_json.get("motivo_analisis", "Sin motivo"))
        modelos.append(res_json.get("modelo_interes", "No especifica"))
        respuestas_bot.append(res_json.get("respuesta_sugerida", "Hola, en breve un asesor te contactará."))
    except Exception as e:
        clasificaciones_ia.append("Lead Frío")
        motivos.append(f"Error procesando fila: {e}")
        modelos.append("No especifica")
        respuestas_bot.append("Hola, un asesor comercial se comunicará contigo pronto.")

    # Pausa obligatoria para respetar la cuota de la capa gratuita
    time.sleep(1)

# Inyectamos los análisis en el DataFrame de Pandas
df_clientes['clasificacion_ia'] = clasificaciones_ia
df_clientes['motivo_analisis'] = motivos
df_clientes['modelo_interes'] = modelos
df_clientes['respuesta_sugerida'] = respuestas_bot

# Exportamos el archivo CSV final
archivo_final = 'mensajes_clasificados_ia.csv'
df_clientes.to_csv(archivo_final, index=False, encoding='utf-8-sig')

print(f"\n🏆 ¡PROCESO FINALIZADO CON ÉXITO!")
print(f"💾 Archivo guardado localmente como '{archivo_final}'")
print("📥 Iniciando descarga automática a tu computadora...")

# Descarga inmediata a través de tu navegador
files.download(archivo_final)

🚀 Iniciando el pipeline analítico modernizado de R&V Motor...
✅ Conexión segura con el nuevo SDK de Google Gemini establecida.

⏳ Fase 1: Generando lote de 50 mensajes sintéticos de clientes...
💾 Datos base de la Semana 1 generados internamente con éxito.

🤖 Fase 2: Configurando el Agente de IA y procesando la clasificación...
⏳ Procesando los mensajes fila por fila con el Agente LLM...

🏆 ¡PROCESO FINALIZADO CON ÉXITO!
💾 Archivo guardado localmente como 'mensajes_clasificados_ia.csv'
📥 Iniciando descarga automática a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>